In [ ]:
import optuna

study_name = ""  # replace with your actual study name
# dataset = "Simulations_indep_traincontrol"
dataset = "NCT00113763"
n_trials = 150
# optuna_version_name = "ExMetrics2_seedData{}_seedHPO{}".format(0, 10)
optuna_version_name = "ExMetrics2_seedHPO{}".format(10)
n_samples = 600
n_features_bytype = 6
treatment_effect = 0.
name_config = "simu_N{}_nfeat{}_t{}".format(n_samples, n_features_bytype, int(treatment_effect))
generator_name = "HI-VAE_weibull" # "HI-VAE_weibull" # "HI-VAE_piecewise" 
# study_name_cluster = "/home/pchassat/survgen-clinical-trials/dataset/{}/optuna_results/optuna_study_{}_ntrials{}_{}_{}".format(dataset, name_config, n_trials, optuna_version_name, generator_name)
# db_file = "/Users/pchassat/Documents/survgen-clinical-trials/dataset/{}/optuna_results/optuna_study_{}_ntrials{}_{}_{}.db".format(dataset, name_config, n_trials, optuna_version_name, generator_name)
study_name_cluster = "/home/pchassat/survgen-clinical-trials/dataset/{}/optuna_results/optuna_study_traincontrol_{}_ntrials{}_{}_{}".format(dataset, dataset, n_trials, optuna_version_name, generator_name)
db_file = "/Users/pchassat/Documents/survgen-clinical-trials/dataset/{}/optuna_results/optuna_study_traincontrol_{}_ntrials{}_{}_{}.db".format(dataset, dataset, n_trials, optuna_version_name, generator_name)
storage = f"sqlite:///{db_file}"
study = optuna.load_study(study_name=study_name_cluster, storage=storage)
names_objs = ["Survival curves dist","Identifiability score"]

In [17]:
from optuna.visualization import plot_parallel_coordinate, plot_slice, plot_param_importances

for i in range(len(names_objs)):
    plot_parallel_coordinate(study, target=lambda t: t.values[i], target_name=names_objs[i]).show() # relationships between objectives and parameters

In [18]:
for i in range(len(names_objs)):
    plot_slice(study, target=lambda t: t.values[i], target_name=names_objs[i]).show() # individual parameter effects

In [19]:
for i in range(len(names_objs)):
    print(f"Parameter importances for {names_objs[i]}:")
    plot_param_importances(study, target=lambda t: t.values[i], target_name=names_objs[i]).show() 

Parameter importances for Survival curves dist:


Parameter importances for Identifiability score:


In [20]:
# Get all trials on the Pareto front
pareto_trials = study.best_trials  # these are Pareto optimal

for t in pareto_trials:
    print("Values:", t.values)
    print("Params:", t.params)
    print("------")

Values: [0.0080066801319753, 0.47368421052631576]
Params: {'lr': 0.002, 'batch_size': 285, 'z_dim': 120, 'y_dim': 130, 's_dim': 50, 'n_layers_surv_piecewise': 1, 'n_intervals': 20}
------
Values: [0.008315847771235046, 0.47157894736842104]
Params: {'lr': 0.005, 'batch_size': 228, 'z_dim': 130, 'y_dim': 50, 's_dim': 20, 'n_layers_surv_piecewise': 1, 'n_intervals': 15}
------
Values: [0.008337592364434796, 0.4589473684210526]
Params: {'lr': 0.0002, 'batch_size': 32, 'z_dim': 50, 'y_dim': 100, 's_dim': 90, 'n_layers_surv_piecewise': 1, 'n_intervals': 20}
------
Values: [0.012192027187010571, 0.43157894736842106]
Params: {'lr': 0.005, 'batch_size': 228, 'z_dim': 70, 'y_dim': 170, 's_dim': 200, 'n_layers_surv_piecewise': 2, 'n_intervals': 20}
------
Values: [0.009168801934496372, 0.4421052631578947]
Params: {'lr': 0.002, 'batch_size': 95, 'z_dim': 200, 'y_dim': 40, 's_dim': 190, 'n_layers_surv_piecewise': 2, 'n_intervals': 20}
------


In [21]:
import pandas as pd

# df_pareto = pd.DataFrame([
#     {**t.params, **{names_objs[i]: v for i, v in enumerate(t.values)}}
#     for t in study.best_trials
# ])
# print(df_pareto)

df_pareto = pd.DataFrame([
    {
        "trial_number": t.number,          
        **t.params,
        **{names_objs[i]: v for i, v in enumerate(t.values)}
    }
    for t in study.best_trials
])

df_pareto.head(10)

,trial_number,lr,batch_size,z_dim,y_dim,s_dim,n_layers_surv_piecewise,n_intervals,Survival curves dist,Identifiability score
0,34,0.0020,285,120,130,50,1,20,0.008007,0.473684
1,46,0.0050,228,130,50,20,1,15,0.008316,0.471579
2,47,0.0002,32,50,100,90,1,20,0.008338,0.458947
3,86,0.0050,228,70,170,200,2,20,0.012192,0.431579
4,126,0.0020,95,200,40,190,2,20,0.009169,0.442105


In [22]:
from optuna.visualization import plot_optimization_history

for i in range(len(names_objs)):
    plot_optimization_history(study, target=lambda t: t.values[i], target_name=names_objs[i]).show() 

In [23]:
from optuna.trial import TrialState
completed = [t for t in study.trials if t.state == TrialState.COMPLETE]
print("Number of trials:", len(study.trials))
print("Number of completed trials:", len(completed))

Number of trials: 150
Number of completed trials: 150


In [25]:
from optuna.visualization import plot_pareto_front
plot_pareto_front(
    study,
    targets=lambda t: [t.values[0], t.values[1]],
    target_names=["Survival curves dist", "Identifiability score"],
    include_dominated_trials=True
)

In [28]:
selected_trial_id = 86
best_trial = study.trials[selected_trial_id]

print("Values (objectifs):", best_trial.values)
print("Params:", best_trial.params)
print("State:", best_trial.state)
print("Start:", best_trial.datetime_start)
print("End:", best_trial.datetime_complete)
print("Duration:", best_trial.duration)

Values (objectifs): [0.012192027187010571, 0.43157894736842106]
Params: {'lr': 0.005, 'batch_size': 228, 'z_dim': 70, 'y_dim': 170, 's_dim': 200, 'n_layers_surv_piecewise': 2, 'n_intervals': 20}
State: 1
Start: 2026-06-16 16:23:39.007141
End: 2026-06-16 16:24:50.536112
Duration: 0:01:11.528971


### Save best selected trial

In [29]:
import json
parent_path = "/Users/pchassat/Documents/survgen-clinical-trials"
# best_params_file = parent_path + "/dataset/" + dataset + "/optuna_results/best_params_{}_ntrials{}_{}_{}.json".format(name_config, n_trials, optuna_version_name, generator_name)
best_params_file = parent_path + "/dataset/" + dataset + "/optuna_results/best_params_traincontrol_{}_ntrials{}_{}_{}.json".format(dataset, n_trials, optuna_version_name, generator_name)
with open(best_params_file, "w") as f:
    json.dump(best_trial.params, f)